In [1]:
import os
import re
import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from collections import Counter
from tqdm.notebook import tqdm
import pickle

DATASET_PATH = '/home/shizm/DL_LABs/DL-lab6/data'  # поправьте путь если нужно
N_FRAMES = 8
IMG_SIZE = 224
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RANDOM_SEED = 42

In [2]:
# Ячейка 2: Сортировка имён файлов
def numerical_sort_key(filename):
    nums = re.findall(r'\d+', filename)
    return int(''.join(nums)) if nums else filename

In [6]:
# Ячейка 3: Загрузка модели ConvNeXt и трансформаций
model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
model.classifier[-1] = nn.Identity()   # убираем классификатор, оставляем эмбеддинги 768
model.eval().to(DEVICE)

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [8]:
# Ячейка 4: Функции для работы с кадрами и извлечения признаков
def get_sorted_frames(seq_path):
    """Возвращает отсортированные полные пути к изображениям."""
    files = [f for f in os.listdir(seq_path) if f.lower().endswith(('.png','.jpg','.jpeg'))]
    files.sort(key=numerical_sort_key)
    return [os.path.join(seq_path, f) for f in files]

def select_frames(frame_list, n=N_FRAMES):
    """Равномерно выбирает n кадров. Если меньше n – дублирует последний."""
    L = len(frame_list)
    if L <= n:
        return frame_list + [frame_list[-1]] * (n - L)
    idx = np.linspace(0, L-1, n, dtype=int)
    return [frame_list[i] for i in idx]

def extract_sequence_features(frame_paths):
    feats = []
    with torch.no_grad():
        for p in frame_paths:
            img = cv2.imread(p)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            t = transform(img).unsqueeze(0).to(DEVICE)
            # Извлекаем признаки и делаем flatten
            f = model(t).cpu().numpy().flatten()
            feats.append(f)
            
    feats = np.array(feats) # размер (8, 768)
    
    # 1. Пространственный контекст (какие объекты есть в кадре: тележка, коробка)
    mean_feat = np.mean(feats, axis=0)
    
    # 2. Интенсивность движения (Стандартное отклонение)
    # Если человек стоит (inaction), дисперсия будет близка к 0. Если идет (move) - будет высокой.
    std_feat = np.std(feats, axis=0)
    
    # 3. Направление действия (Разница между последним и первым кадром)
    diff_feat = feats[-1] - feats[0]
    
    # Склеиваем всё вместе: размер итогового вектора станет 2304
    return np.concatenate([mean_feat, std_feat, diff_feat])

In [9]:
# Ячейка 5: Обход датасета и формирование признаков
X, y, paths = [], [], []

for cls in ['inaction', 'move', 'work']:
    cls_dir = os.path.join(DATASET_PATH, cls)
    if not os.path.isdir(cls_dir):
        continue
    for root, dirs, files in os.walk(cls_dir):
        if root == cls_dir:
            continue
        frames = get_sorted_frames(root)
        if not frames:
            continue
        selected = select_frames(frames)
        feat = extract_sequence_features(selected)
        X.append(feat)
        y.append(cls)
        paths.append(root)

X = np.array(X)
y = np.array(y)
print(f"Всего примеров: {len(X)}")
print("Распределение классов:", Counter(y))

Всего примеров: 585
Распределение классов: Counter({np.str_('work'): 342, np.str_('move'): 150, np.str_('inaction'): 93})


In [10]:
# Ячейка 6: Кодирование меток и разделение train/val
label_to_idx = {cls: i for i, cls in enumerate(sorted(set(y)))}
idx_to_label = {i: cls for cls, i in label_to_idx.items()}
y_int = np.array([label_to_idx[cls] for cls in y])

X_train, X_val, y_train, y_val = train_test_split(
    X, y_int,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y_int
)

print(f"Train size: {len(X_train)}, Val size: {len(X_val)}")
print("Train class dist:", Counter(idx_to_label[i] for i in y_train))
print("Val class dist:", Counter(idx_to_label[i] for i in y_val))

Train size: 468, Val size: 117
Train class dist: Counter({np.str_('work'): 274, np.str_('move'): 120, np.str_('inaction'): 74})
Val class dist: Counter({np.str_('work'): 68, np.str_('move'): 30, np.str_('inaction'): 19})


In [11]:
# Ячейка 7: Сохранение
np.savez('prepared_data_convnext.npz',
         X_train=X_train, y_train=y_train,
         X_val=X_val, y_val=y_val,
         label_to_idx=label_to_idx, idx_to_label=idx_to_label)

with open('class_mapping_convnext.pkl', 'wb') as f:
    pickle.dump({'label_to_idx': label_to_idx, 'idx_to_label': idx_to_label}, f)

print("Данные сохранены в prepared_data_convnext.npz и class_mapping_convnext.pkl")

Данные сохранены в prepared_data_convnext.npz и class_mapping_convnext.pkl
